# Chapter 00 — GPU Benchmark: FP32 vs FP16 vs BF16 Matmul

Large language models are, at their core, compute-bound workloads. Before writing a single line of model code, you need to understand the hardware that will execute it. This chapter is the true prerequisite: it maps the physical infrastructure — GPUs, memory, interconnects, and cloud clusters — that determines whether your training run takes hours or weeks.

### Glossary / Glossário

• GPU — Graphics Processing Unit, specialized processor for parallel computation, essential for training neural networks. / Processador especializado em computação paralela, essencial para treinar redes neurais.
• CUDA — Compute Unified Device Architecture, NVIDIA's parallel computing platform for general-purpose GPU programming. / Plataforma de computação paralela da NVIDIA para programação de propósito geral em GPUs.
• Tensor Cores — specialized GPU units optimized for matrix multiply-accumulate operations. / Unidades especializadas da GPU otimizadas para operações de multiplicação-acumulação de matrizes.
• TFLOPS — Tera Floating-Point Operations Per Second, a measure of GPU throughput. / Trilhões de operações de ponto flutuante por segundo, medida de vazão da GPU.
• VRAM — Video Random Access Memory, the GPU's dedicated memory for model weights and data. / Memória dedicada da GPU para pesos do modelo e dados.
• HBM — High Bandwidth Memory, stacked memory technology used in data-center GPUs. / Memória de alta largura de banda, tecnologia de memória empilhada usada em GPUs de datacenter.
• FP32 — 32-bit floating point, full precision format (4 bytes per number). / Ponto flutuante de 32 bits, formato de precisão total (4 bytes por número).
• FP16 — 16-bit floating point, half precision format that uses half the memory of FP32. / Ponto flutuante de 16 bits, formato de meia precisão que usa metade da memória do FP32.
• BF16 — Brain Float 16, 16-bit format that keeps FP32's dynamic range while saving memory. / Brain Float 16, formato de 16 bits que mantém a faixa dinâmica do FP32 economizando memória.
• NVLink — NVIDIA's high-speed GPU-to-GPU interconnect, much faster than PCIe. / Interconexão GPU-a-GPU de alta velocidade da NVIDIA, muito mais rápida que PCIe.
• NCCL — NVIDIA Collective Communication Library for efficient multi-GPU operations. / Biblioteca de comunicação coletiva da NVIDIA para operações multi-GPU eficientes.
• PCIe — Peripheral Component Interconnect Express, standard bus connecting GPUs to CPU. / Barramento padrão que conecta GPUs à CPU.
• matmul — matrix multiplication, the core operation underlying neural network computations. / Multiplicação de matrizes, operação fundamental das computações em redes neurais.

> **Setup:** This notebook requires a GPU runtime. In Colab, go to **Runtime → Change runtime type → T4 GPU** (or any available GPU).

In [ ]:
import torch
import time

def get_gpu_info():
    """Get GPU info using PyTorch CUDA APIs."""
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No GPU found! In Colab, go to Runtime → Change runtime type → T4 GPU"
        )
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)  # GPU hardware properties (name, memory, compute capability)
        mem_total = props.total_mem / 1e6  # total VRAM in megabytes
        mem_free = (props.total_mem - torch.cuda.memory_allocated(i)) / 1e6  # available VRAM in megabytes
        print(f"GPU {i}: {props.name} | {mem_total:.0f} MB total | {mem_free:.0f} MB free")

def benchmark_matmul(size=4096, dtype=torch.float32, warmup=10, iters=100):  # size = matrix dimension (N×N), dtype = numerical precision format
    """Benchmark matrix multiplication at a given precision."""
    device = torch.device("cuda")
    a = torch.randn(size, size, dtype=dtype, device=device)
    b = torch.randn(size, size, dtype=dtype, device=device)

    # Warmup
    for _ in range(warmup):
        torch.mm(a, b)
    torch.cuda.synchronize()

    # Timed iterations
    start = time.perf_counter()
    for _ in range(iters):
        torch.mm(a, b)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    tflops = (2 * size**3 * iters) / (elapsed * 1e12)  # trillion floating-point operations per second (throughput)
    mem_mb = torch.cuda.max_memory_allocated() / 1e6  # peak GPU memory allocated in megabytes
    print(f"{str(dtype):20s} | {elapsed/iters*1000:8.2f} ms/iter | {tflops:6.1f} TFLOPS | {mem_mb:8.1f} MB peak")

if __name__ == "__main__":
    get_gpu_info()
    print("\n--- Matrix Multiplication Benchmark (4096x4096) ---")
    print(f"{'Dtype':20s} | {'Time':>12s} | {'TFLOPS':>10s} | {'Memory':>12s}")
    print("-" * 65)
    for dtype in [torch.float32, torch.float16, torch.bfloat16]:
        torch.cuda.reset_peak_memory_stats()
        benchmark_matmul(dtype=dtype)

    print("\n=== Key Takeaway ===")
    print("FP16/BF16 use ~half the memory of FP32 and are typically 2-4x faster on Tensor Core GPUs!")

---

**Course:** Aprenda LLM | **Chapter 00**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.